In [146]:
import pandas as pd
import spacy

nlp = spacy.load("en_core_web_sm")

df = pd.read_csv("datathon_year_3.csv")

# rows, cols
print(df.shape)
# types
print(df.dtypes)

(3995, 15)
Idea #                                int64
Original Title                          str
Challenge                               str
Solution                                str
Audience                                str
Borough                                 str
Neighborhood(s)                         str
Host of Idea Generation Workshop        str
Impact Area                             str
Sub-Category                            str
Type of Idea                            str
Idea Zip Code                       float64
Idea Generation Session Zip Code    float64
Idea Source                             str
Insights                                str
dtype: object


In [147]:
# preview of first 2 rows
print(df.head(2))

   Idea #                          Original Title  \
0      14  Pathways to Success: Bilingual Support   
1     111                    Youth Launch Project   

                                           Challenge  \
0  Teens of immigrants struggle with the college ...   
1  A challenge in Harlem is helping students lear...   

                                            Solution    Audience    Borough  \
0  A mentorship program that would offer bilingua...  Immigrants     Queens   
1  A youth enterprenuership program that teaches ...       Youth  Manhattan   

  Neighborhood(s)                   Host of Idea Generation Workshop  \
0             NaN  [10/10/2024 @ 2:00 PM] DOE Civics For All Even...   
1          Harlem  [10/15/2024 @ 5:30 PM] Lee Heritage Media LLC ...   

  Impact Area                                       Sub-Category  \
0   Education  EDU - Adult Educational Programs,EDU - Aftersc...   
1   Education  EDU - Professional Skills Development,EDU - Jo...   

           

In [148]:
# drop unnecessary columns 
df = df.drop(['Host of Idea Generation Workshop', 'Idea Zip Code', 'Idea Generation Session Zip Code', 'Idea Source'], axis = 1)
print(df.dtypes)

Idea #             int64
Original Title       str
Challenge            str
Solution             str
Audience             str
Borough              str
Neighborhood(s)      str
Impact Area          str
Sub-Category         str
Type of Idea         str
Insights             str
dtype: object


In [149]:
# check for NA values
print(df.isna().sum())

Idea #               0
Original Title      41
Challenge            6
Solution            13
Audience             1
Borough              0
Neighborhood(s)    201
Impact Area          4
Sub-Category       474
Type of Idea       703
Insights           838
dtype: int64


In [ ]:
# Check rows where Challenge or Solution is duplicated
duplicated_rows = df[df.duplicated(subset=["Challenge"], keep=False) |
                     df.duplicated(subset=["Solution"], keep=False)]

# preview of dupes
print("Duplicated rows:\n", duplicated_rows)

# drop duplicates
df = df.drop_duplicates(subset=["Challenge"], keep='first')
df = df.drop_duplicates(subset=["Solution"], keep='first')

Duplicated rows:
 Empty DataFrame
Columns: [Idea #, Original Title, Challenge, Solution, Audience, Borough, Neighborhood(s), Impact Area, Sub-Category, Type of Idea, Insights]
Index: []
Duplicated rows:
 Empty DataFrame
Columns: [Idea #, Original Title, Challenge, Solution, Audience, Borough, Neighborhood(s), Impact Area, Sub-Category, Type of Idea, Insights]
Index: []


In [152]:
# NaN Check
# If NaN left in data, it changes the column to a float causing errors
df["Challenge"].apply(type).value_counts()

# drop missing values
df = df.dropna(subset=["Challenge", "Solution"])

In [153]:
# remove characters from rows that start with bad chars
df[['Challenge', 'Solution']] = df[['Challenge', 'Solution']].apply(
    lambda col: col.str.lstrip("\"'@-=")
)

# ^^^ above uses anon func lambda to apply the following func into each col, acts as wrapper so can use .apply
# equivalent to:
# def clean_start(col):
#     return col.str.lstrip("\"'@-=")

# df[cols_to_clean] = df[cols_to_clean].apply(clean_start)

# remove "=" from text
# for each selected column, replace characters in all strings
df[['Challenge', 'Solution']] = df[['Challenge', 'Solution']].apply(
    lambda col: col.str.replace("=", "", regex=False))

In [154]:
# noun chunk extraction
def extract_noun_chunk(text):
    doc = nlp(text.lower())
    return " | ".join([chunk.text for chunk in doc.noun_chunks])

df['Challenge_noun_chunk'] = df['Challenge'].apply(extract_noun_chunk)
df['Solution_noun_chunk'] = df['Solution'].apply(extract_noun_chunk)
temp_df_nc = df[['Challenge_noun_chunk', 'Solution_noun_chunk']]
print(temp_df_nc.head(10))

                                Challenge_noun_chunk  \
0  teens | immigrants | the college application p...   
1  a challenge | harlem | students | their own bu...   
2  some seniors | limited credit | no income | ho...   
3           many teens | methods | income | canarsie   
4  many younger parents | parents | no help | par...   
5  schools | a rigid structure | that | almost no...   
6  no clubs | elementary school grades | high sch...   
7  pakistani women immigrants | no support form p...   
8  lack | “relevant” work opportunities | teens/y...   
9                                      people | they   

                                 Solution_noun_chunk  
0  a mentorship program | that | bilingual worksh...  
1  a youth enterprenuership program | that | stud...  
2  an urgent need | senior apartments | queens | ...  
3  a program | teens | available resources | empl...  
4  more parenting coaching classes | young adults...  
5  programs | that | host clubs | students | know... 

In [155]:
# lemmatize function
def clean_lemmatize(text):
    doc = nlp(text.lower())
    return " ".join([
        token.lemma_
        for token in doc
        # punctuation   
        if not token.is_punct   
        # spaces 
        and not token.is_space   
        # stopwords
        and not token.is_stop  
    ])

df['Challenge_lem'] = df['Challenge'].apply(clean_lemmatize)
df['Solution_lem'] = df['Solution'].apply(clean_lemmatize)
lem_df = df[['Challenge_lem', 'Solution_lem']]
print(lem_df.head(20))

                                        Challenge_lem  \
0   teen immigrant struggle college application pr...   
1   challenge harlem help student learn start busi...   
2   senior limited credit income find housing majo...   
3                         teen method income canarsie   
4                young parent parent help grow parent   
5   school rigid structure leave time extra curric...   
6   club elementary school grade like high school ...   
7   pakistani woman immigrant unemployed support f...   
8   lack relevant work opportunity teen youth word...   
9                              people drown know swim   
10  lack access free financial literacy education ...   
11  new york city housing crisis landlord advantag...   
12                youth outlet express gift misdirect   
13                              people cook mean feed   
14  folk limited access housing struggle food inse...   
15  youth employment trade programming outreach of...   
16  awareness autism student di

In [156]:
# dependency parsing

# def show_dependencies(text):
#     doc = nlp(text)
#     for token in doc:
#         print(token.text, token.dep_, token.head.text)

# subject object verb extraction, including prepositional objs
def extract_svo(text):
    doc = nlp(text)
    svos = []
    
    for token in doc:
        if token.pos_ == "VERB":
            subjects = [child for child in token.children if child.dep_ in ["nsubj", "nsubjpass"]]
            objects = [child for child in token.children if child.dep_ in ["dobj", "attr", "oprd"]]
            
            # include prepositional objects
            for prep in [child for child in token.children if child.dep_ == "prep"]:
                objects.extend([child for child in prep.children if child.dep_ == "pobj"])
            
            for subj in subjects:
                for obj in objects:
                    svos.append((subj.text, token.lemma_, obj.text))
    
    return svos

df["svos_problems"] = df["Challenge"].apply(extract_svo)
df["svos_solutions"] = df["Solution"].apply(extract_svo)

svo_df = df[['svos_problems', 'svos_solutions']]
print(svo_df.head(20))

# test for checking visualization of SVO on row 5 (cant use on whole column)
# for text in df["Community problems"].head(5):
#     doc = nlp(text)
#     displacy.render(doc, style="dep", jupyter=True)

                                        svos_problems  \
0   [(Teens, struggle, processes), (parents, advis...   
1                                                  []   
2                                                  []   
3                            [(teens, have, methods)]   
4                          [(parents, grow, parents)]   
5   [(Schools, have, structure), (that, leave, tim...   
6   [(This, offer, experiences), (This, offer, age...   
7                                                  []   
8                                                  []   
9                                                  []   
10                                                 []   
11  [(that, take, advantage), (that, take, people)...   
12                           [(youth, have, outlets)]   
13                                                 []   
14                                                 []   
15              [(employment, oftentime, experience)]   
16             [(child, have, I

In [ ]:
# most common issues from challenge lem
from collections import Counter

all_words = " ".join(df["Challenge_lem"]).split()
word_counts = Counter(all_words).most_common(50)
df_counts = pd.DataFrame(word_counts, columns=["Common Issue", "Count"])
print(df_counts)

   Common Issue  Count
0        people   1452
1     community    960
2          need    834
3         youth    629
4          lack    588
5       program    539
6        school    529
7          help    516
8       housing    463
9           job    350
10        child    341
11       health    332
12    immigrant    322
13        adult    295
14        young    292
15       mental    291
16    challenge    290
17      problem    285
18          low    285
19      support    281
20         food    271
21       senior    269
22       parent    267
23       income    267
24     resource    267
25       access    266
26      english    266
27       street    265
28         like    251
29       public    248


In [ ]:
# most common issues from noun chunk challenges
all_chunks = []
for chunks in df["Challenge_noun_chunk"]:
    all_chunks.extend(chunks.split(" | "))

Counter(all_chunks).most_common(50)

[('they', 782),
 ('people', 719),
 ('that', 619),
 ('i', 579),
 ('we', 485),
 ('it', 430),
 ('them', 374),
 ('who', 354),
 ('lack', 329),
 ('youth', 232),
 ('the community', 211),
 ('a lot', 188),
 ('immigrants', 163),
 ('school', 159),
 ('which', 145),
 ('the youth', 142),
 ('children', 141),
 ('disabilities', 136),
 ('parents', 130),
 ('housing', 130),
 ('seniors', 130),
 ('access', 128),
 ('this', 123),
 ('resources', 114),
 ('help', 114),
 ('the streets', 113),
 ('young people', 110),
 ('english', 108),
 ('older adults', 104),
 ('kids', 100)]

In [ ]:
# SVO relationship analysis
# singular verbs freq
verbs = []
for row in df["svos_problems"]:
    for (_, verb, _) in row:
        verbs.append(verb)

Counter(verbs).most_common(50)

[('have', 708),
 ('need', 419),
 ('see', 197),
 ('get', 138),
 ('face', 127),
 ('go', 77),
 ('take', 72),
 ('live', 67),
 ('lack', 65),
 ('use', 63),
 ('provide', 58),
 ('speak', 58),
 ('leave', 54),
 ('help', 54),
 ('lead', 52),
 ('afford', 48),
 ('do', 48),
 ('come', 46),
 ('know', 46),
 ('offer', 44),
 ('struggle', 41),
 ('impact', 41),
 ('give', 40),
 ('experience', 39),
 ('affect', 38),
 ('make', 37),
 ('find', 36),
 ('create', 33),
 ('spend', 31),
 ('learn', 29),
 ('suffer', 28),
 ('include', 27),
 ('receive', 25),
 ('become', 25),
 ('feel', 24),
 ('keep', 24),
 ('understand', 23),
 ('teach', 23),
 ('like', 22),
 ('pay', 22),
 ('support', 20),
 ('limit', 20),
 ('benefit', 20),
 ('work', 20),
 ('lose', 19),
 ('put', 19),
 ('address', 19),
 ('cause', 18),
 ('deserve', 16),
 ('engage', 16)]

In [162]:
# Subject–verb, verb–object 
pairs = []
for row in df["svos_problems"]:
    for (subj, verb, obj) in row:
        pairs.append((verb, obj))

Counter(pairs).most_common(50)

[(('need', 'help'), 59),
 (('see', 'community'), 54),
 (('have', 'access'), 41),
 (('have', 'time'), 35),
 (('see', 'that'), 31),
 (('speak', 'English'), 28),
 (('face', 'challenges'), 22),
 (('need', 'programs'), 20),
 (('need', 'support'), 18),
 (('have', 'resources'), 17),
 (('have', 'opportunity'), 16),
 (('have', 'programs'), 15),
 (('have', 'place'), 15),
 (('get', 'help'), 15),
 (('have', 'experience'), 13),
 (('need', 'assistance'), 12),
 (('have', 'lot'), 12),
 (('have', 'money'), 12),
 (('spend', 'time'), 11),
 (('see', 'people'), 11),
 (('have', 'trouble'), 11),
 (('need', 'activities'), 11),
 (('see', 'lot'), 11),
 (('have', 'space'), 10),
 (('have', 'opportunities'), 10),
 (('have', 'problem'), 10),
 (('lack', 'access'), 9),
 (('impact', 'people'), 9),
 (('have', 'support'), 9),
 (('have', 'difficulty'), 9),
 (('have', 'knowledge'), 9),
 (('face', 'barriers'), 9),
 (('take', 'care'), 8),
 (('have', 'skills'), 8),
 (('need', 'space'), 8),
 (('have', 'issues'), 8),
 (('need'

In [ ]:
# problem-to-solution column mapping
mapping = []

for i, row in df.iterrows():
    for (subj, verb, obj) in row["svos_problems"]:
        for (s_subj, s_verb, s_obj) in row["svos_solutions"]:
            mapping.append((verb + " " + obj, s_verb + " " + s_obj))

Counter(mapping).most_common(50)

[(('leave terms', 'offer programs'), 18),
 (('leave dust', 'offer programs'), 18),
 (('stem challenges', 'offer programs'), 18),
 (('lack training', 'offer programs'), 13),
 (('retain character', 'secure employment'), 12),
 (('miss activities', 'offer programs'), 10),
 (('retain character', 'train residents'), 9),
 (('advance speeds', 'offer programs'), 9),
 (('release functionalities', 'offer programs'), 9),
 (('respond city', 'offer programs'), 9),
 (('experience disparities', 'offer programs'), 9),
 (('take that', 'offer programs'), 9),
 (('live lack', 'offer programs'), 9),
 (('live inequities', 'offer programs'), 9),
 (('have quality', 'offer programs'), 9),
 (('exist communities', 'offer programs'), 9),
 (('widen pace', 'offer programs'), 9),
 (('acquire age', 'offer programs'), 9),
 (('lose time', 'offer programs'), 9),
 (('exist NYC', 'offer programs'), 9),
 (('lack access', 'offer programs'), 9),
 (('make strides', 'offer programs'), 9),
 (('accomplish what', 'offer programs')

In [ ]:
# Term Frequency-Inverse Document Frequency (TF-IDF)
# TF-IDF score is how important a word or phrase is across data
# Score high when term frequent in some responses, but is not everywhere
# AKA not "most common", "most severe", etc...
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

vectorizer = TfidfVectorizer(
    stop_words='english',
    token_pattern=r'\b[a-zA-Z]{3,}\b',
    min_df=3,
    max_df=0.8,
    ngram_range=(1, 2)
)

X = vectorizer.fit_transform(df['Challenge_lem'])

feature_names = vectorizer.get_feature_names_out()

# compute average TF-IDF scores
scores = np.asarray(X.mean(axis=0)).ravel()

# get top 30 terms
top_indices = scores.argsort()[::-1][:30]

top_terms_df = pd.DataFrame({
    "term": [feature_names[i] for i in top_indices],
    "score": [round(float(scores[i]), 4) for i in top_indices]
})
print(top_terms_df)

         term   score
0      people  0.0394
1   community  0.0272
2        need  0.0257
3       youth  0.0246
4     program  0.0205
5        lack  0.0204
6     housing  0.0195
7        help  0.0193
8      school  0.0187
9         job  0.0146
10  immigrant  0.0144
11      child  0.0134
12     senior  0.0134
13     health  0.0133
14      adult  0.0131
15    problem  0.0127
16     mental  0.0126
17       food  0.0125
18    english  0.0122
19      young  0.0119
20     street  0.0119
21        old  0.0117
22     parent  0.0117
23     income  0.0116
24   resource  0.0115
25     public  0.0113
26        low  0.0112
27        lot  0.0108
28   homeless  0.0107
29  challenge  0.0105


In [ ]:
# clustering
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=5)
kmeans.fit(X)

df["cluster"] = kmeans.labels_

In [159]:
# output
# in-notebook display
df

# csv export
# df.to_csv("output.csv", index=False)

,Idea #,Original Title,Challenge,Solution,Audience,Borough,Neighborhood(s),Impact Area,Sub-Category,Type of Idea,Insights,Challenge_noun_chunk,Solution_noun_chunk,Challenge_lem,Solution_lem,svos_problems,svos_solutions
0,14,Pathways to Success: Bilingual Support,Teens of immigrants struggle with the college ...,A mentorship program that would offer bilingua...,Immigrants,Queens,NaN,Education,"EDU - Adult Educational Programs,EDU - Aftersc...","""Classes, Workshop or Training""","Problem,Solution",teens | immigrants | the college application p...,a mentorship program | that | bilingual worksh...,teen immigrant struggle college application pr...,mentorship program offer bilingual workshop se...,"[(Teens, struggle, processes), (parents, advis...","[(that, offer, workshops), (We, train, mentors..."
1,111,Youth Launch Project,A challenge in Harlem is helping students lear...,A youth enterprenuership program that teaches ...,Youth,Manhattan,Harlem,Education,"EDU - Professional Skills Development,EDU - Jo...","""Classes, Workshop or Training""",Solution,a challenge | harlem | students | their own bu...,a youth enterprenuership program | that | stud...,challenge harlem help student learn start busi...,youth enterprenuership program teach student s...,[],"[(that, teach, students)]"
2,209,Housing workshop for seniors,For some seniors with limited credit and no in...,"There is an urgent need for senior apartments,...","Older Adults,Unhoused People,Low Income People",Queens,Queens (Flushing & Bayside),"Other,Social Services & Accessiblity",SS - Housing Assistance,Service Delivery,"Problem,Solution",some seniors | limited credit | no income | ho...,an urgent need | senior apartments | queens | ...,senior limited credit income find housing majo...,urgent need senior apartment particularly quee...,[],[]
3,289,Teen Employee Connect (TEC),Many teens do not have methods of income in Ca...,develop a program to connect teens with availa...,"Youth,Low Income People",Brooklyn,"Canarsie, Flatbush",Workforce Development,"EDU - Professional Skills Development,WD - Job...","""Classes, Workshop or Training""",Solution,many teens | methods | income | canarsie,a program | teens | available resources | empl...,teen method income canarsie,develop program connect teen available resourc...,"[(teens, have, methods)]",[]
4,348,"""Your not alone""",Many younger parents and parents with no help ...,More parenting coaching classes for young adul...,Parents,Brooklyn,"Brownsville, Brooklyn",Social Services & Accessiblity,SS - Parenting Support & Prevention,"""Classes, Workshop or Training""","Problem,Solution",many younger parents | parents | no help | par...,more parenting coaching classes | young adults...,young parent parent help grow parent,parenting coaching class young adult support h...,"[(parents, grow, parents)]","[(parents, gain, knowledge), (parents, gain, t..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3990,4334,Bridging the Exponentially Growing Digital Div...,Technology often advances at breakneck speeds ...,This project proposes a multifaceted approach ...,"Youth,Older Adults,Public Housing Residents,Ju...",Bronx,All areas.,Education,NaN,NaN,NaN,technology | breakneck speeds | this | the rea...,this project | a multifaceted approach | the a...,technology advance breakneck speed especially ...,project propose multifaceted approach bridge a...,"[(Technology, advance, speeds), (which, releas...","[(project, propose, approach), (We, foster, pa..."
3991,4509,Auxiliary Police Recruitment,Most people are opting to work from home and s...,"Hire 10 Auxiliary police, put them through a s...",Someone Else,Manhattan,This should be for major MTA depos in Brooklyn...,Public Safety,PS - Community Safety,Service Delivery,"Problem,Solution",most people | home | they | safe commuting | t...,10 auxiliary police | them | a safety voluntee...,people opt work home shop online feel safe com...,hire 10 auxiliary police safety volunteer trai...,"